# Knee — training on Colab free (T4)

Runs **the same generated trainer** as the Kaggle kernels — `kaggle/<n>/run.py`,
byte-identical, no reimplementation. Only the paths and the machine change, so a
fold trained here is comparable to one trained on Kaggle. That is the whole
point: a Colab-specific training loop would make every past number
incomparable.

**The rules permit this.** `docs/FINDINGS.md` 2.7, from the Code Requirements
page: *"Freely and publicly available external data and pre-trained models are
allowed."* Only **inference** must run in a Kaggle notebook. Weights come back
in as a private Kaggle Dataset.

**Runtime → Change runtime type → T4 GPU** before running anything.

Costs, measured on Kaggle's 2x T4: **~1.4 h per resnet34 fold**. Colab free
gives one T4, so expect **~2.5-3 h per fold** and run **one fold per session**.


## 1. Check the GPU

If this reports no GPU, fix the runtime type before going further.

In [ ]:
!nvidia-smi
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))


## 2. Mount Drive

Everything persistent lives here: the ~10 GB cache, and the checkpoints. Colab
free disconnects, and re-downloading 10 GB every session is the main thing that
makes this painful. Download once, reuse forever.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/knee')
CACHE = ROOT/'cache'; LABELS = ROOT/'labels'; OUT = ROOT/'runs'; CODE = ROOT/'code'
for d in (CACHE, LABELS, OUT, CODE): d.mkdir(parents=True, exist_ok=True)
print(ROOT, "ready")
!df -h /content/drive/MyDrive | tail -1


## 3. Kaggle credentials

Paste the token into Colab's **Secrets** panel (the key icon, left sidebar) as
`KAGGLE_API_TOKEN` and enable it for this notebook. Do not type it into a cell —
notebook outputs get shared.

In [ ]:
!pip install -q kaggle
from google.colab import userdata
import os
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
pathlib.Path.home().joinpath('.kaggle').mkdir(exist_ok=True)
p = pathlib.Path.home()/'.kaggle'/'access_token'
p.write_text(os.environ['KAGGLE_API_TOKEN']); p.chmod(0o600)
!kaggle kernels list --user achelijndiamantidis --page-size 3


## 4. Get the code

Clones the repo so the trainer is the real generated script, not a copy that can
drift. `generate_kernels.py --check` is the same gate CI runs.

In [ ]:
%cd /content
!rm -rf Knee-abnormality
!git clone -q https://github.com/existentialistlogarithmic/Knee-abnormality.git
%cd /content/Knee-abnormality
!git checkout -q claude/rsna-knee-abnormality-jahn5n
!python eda/generate_kernels.py --check


## 5. Download the cache and labels — once

~10 GB of 192px cache across 4 shards, plus the fused labels and the
competition `train.csv`. Skipped automatically if Drive already has them.

If this dies partway, just re-run the cell — it resumes per shard.

In [ ]:
import subprocess, pathlib
CACHE = pathlib.Path('/content/drive/MyDrive/knee/cache')
LABELS = pathlib.Path('/content/drive/MyDrive/knee/labels')

for shard in range(4):
    d = CACHE/f'shard{shard}'
    if any(d.glob('cache_index_train_*.parquet')):
        print(f'shard {shard}: already present'); continue
    d.mkdir(parents=True, exist_ok=True)
    print(f'shard {shard}: downloading ...')
    subprocess.run(['kaggle','kernels','output',
                    f'achelijndiamantidis/knee-cache-build-{shard}','-p',str(d)], check=True)

if not (LABELS/'soft_labels.parquet').exists():
    subprocess.run(['kaggle','datasets','download',
                    'achelijndiamantidis/knee-phase1-fused','-p',str(LABELS),'--unzip'], check=True)
if not (LABELS/'train.csv').exists():
    subprocess.run(['kaggle','competitions','download',
                    'rsna-knee-abnormality-detection','-f','train.csv','-p',str(LABELS)], check=True)

!du -sh /content/drive/MyDrive/knee/cache
!ls /content/drive/MyDrive/knee/labels


## 6. Stage the cache on local disk

Drive is slow for random reads of many small files, and training reads one
`.npy` per study per epoch. Copying to Colab's local SSD first is worth several
minutes of setup — the alternative is a data-loader-bound run.

Local disk does NOT survive a disconnect; Drive does. That is why both exist.

In [ ]:
import shutil, pathlib, time
SRC = pathlib.Path('/content/drive/MyDrive/knee/cache')
DST = pathlib.Path('/content/cache'); DST.mkdir(exist_ok=True)
t0=time.time()
for shard in sorted(SRC.glob('shard*')):
    for f in shard.rglob('*'):
        if f.is_file():
            target = DST/f.name
            if not target.exists(): shutil.copy2(f, target)
print(f'staged in {(time.time()-t0)/60:.1f} min')
!du -sh /content/cache; ls /content/cache | head -3
!ls /content/cache/*.npy | wc -l


## 7. Train one fold

**Set `FOLD` to the fold you want.** Folds 2, 3 and 4 are the ones still on the
OLD labels (see `docs/HANDOFF.md`) — those are the useful ones to run here.

`--time-budget` is set to 4 h. Colab free sessions get reclaimed; the trainer
writes a checkpoint as it goes, so a killed session loses the tail of a run, not
all of it.

Nothing about the model is set here on purpose: epochs, batch, LR, backbone and
geometry all come from the generated script's own constants, which came from
`src/pipeline.py`. Overriding them from this notebook is how a Colab run would
stop being comparable to a Kaggle one.

In [ ]:
FOLD = 2   # <-- 2, 3 or 4

import pathlib, subprocess, shutil
OUT = pathlib.Path(f'/content/out_fold{FOLD}'); OUT.mkdir(exist_ok=True)
LABELS = '/content/drive/MyDrive/knee/labels'
DIRS = {0:'kaggle/21_train_v1fused', 1:'kaggle/27_train_v1fused_fold1',
        2:'kaggle/28_train_v1fused_fold2', 3:'kaggle/29_train_v1fused_fold3',
        4:'kaggle/30_train_v1fused_fold4'}

cmd = ['python', f'{DIRS[FOLD]}/run.py',
       '--fold', str(FOLD),
       '--cache', '/content/cache',
       '--labels', LABELS,
       '--headers', LABELS,
       '--train-csv', f'{LABELS}/train.csv',
       '--out', str(OUT),
       '--time-budget', str(4*3600)]
print(' '.join(cmd), flush=True)
subprocess.run(cmd, cwd='/content/Knee-abnormality', check=True)


## 8. Save results to Drive

Do this immediately — local disk vanishes when the session ends.

In [ ]:
import shutil, pathlib
FOLD_OUT = pathlib.Path('/content/drive/MyDrive/knee/runs')/f'fold{FOLD}'
FOLD_OUT.mkdir(parents=True, exist_ok=True)
for f in pathlib.Path(f'/content/out_fold{FOLD}').glob('*'):
    if f.is_file(): shutil.copy2(f, FOLD_OUT/f.name)
!ls -la {str(FOLD_OUT)}


## 9. Ship the checkpoints back to Kaggle

Inference must run in a Kaggle notebook with internet off, so the weights have
to arrive as a mounted Dataset. Run this **after** the folds you want are done —
one dataset holding all of them.

Then add `achelijndiamantidis/knee-colab-checkpoints` to
`kaggle/22_infer_v1fused/kernel-metadata.json` as a `dataset_source`, and adjust
the kernel to load from it. **Do not hand-edit `run.py`** — change
`src/pipeline.py` and regenerate, or `generate_kernels.py --check` will fail.

In [ ]:
import json, pathlib, shutil, subprocess
UP = pathlib.Path('/content/upload'); UP.mkdir(exist_ok=True)
runs = pathlib.Path('/content/drive/MyDrive/knee/runs')
for d in sorted(runs.glob('fold*')):
    for f in d.glob('*.pt'):
        shutil.copy2(f, UP/f.name)
    for f in d.glob('gold_oof_*.json'):
        shutil.copy2(f, UP/f.name)

meta = {"title": "Knee — Colab-trained fold checkpoints",
        "id": "achelijndiamantidis/knee-colab-checkpoints",
        "licenses": [{"name": "other"}]}
(UP/'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
print(sorted(p.name for p in UP.iterdir()))

# first time: create. afterwards: version.
NEW = True   # <-- set False after the first upload
if NEW:
    subprocess.run(['kaggle','datasets','create','-p',str(UP),'--dir-mode','zip'])
else:
    subprocess.run(['kaggle','datasets','version','-p',str(UP),'-m','more folds','--dir-mode','zip'])


## 10. Check it against the Kaggle-trained folds

The point of the discipline above is that this comparison is meaningful. A
Colab fold and a Kaggle fold differ only in the machine, so pooling them is
legitimate — **provided they were trained on the same label version**. Folds
trained on different labels must never go into one ensemble; that confound is
recorded in `docs/EXPERIMENTS.md` E040.

In [ ]:
!cd /content/Knee-abnormality && python eda/pool_gold_oof.py \
    /content/drive/MyDrive/knee/runs/fold*/gold_oof_fold*.json
